# Dual-basis (vertex-token) gridinv on Colab — L=4 tuning + primal/dual A/B

**Part 1 — tuning (one cell per run):** training dynamics & architecture for the new
`ToricCNN_gridinv_dual` (vertex-star Wilson coarse-graining on the Hadamard-conjugated
Hamiltonian, branch `feat/dual-basis`), each run an independent cell with **explicit**
`n_iter / dt / lr_min / diag_shift / kernel_size / inv_hidden`, all anchored against a
primal production-config reference at the same point. **Everything logs live to
wandb.ai** (project `approx-sym-3D-TC`; per-step energy/spread/delta, full config incl.
`dual_basis`, timing, final observables, weight artifacts).

**Part 2 — the North Star:** side-by-side primal vs dual at matched parameter count,
same points and seeds. Judged on (i) final energy (lower wins, variational), (ii)
V-score, (iii) stability (spread trajectory, guard rollbacks, divergence). Physical
observables (⟨A_v⟩/⟨B_p⟩/⟨M_z⟩) must AGREE between arms — the dual flag keeps JSON/W&B
keys physical, so disagreement flags a conjugation bug, not physics.

Physics prior (2D dual-basis experiment): performance follows the diagonality of the
perturbing field in the sampling basis. hx-dominated points are the primal net's
off-diagonal regime and the dual net's diagonal regime; the hz point is the control.

Dual-net constraints to keep in mind while tuning: token grid is (L,L,L) with ONE
channel (vs the primal's 3 orientation channels) → ~3.8× fewer params at matched
widths (L=4: primal k3 inv"2 2 2" = 5319; dual k4 inv"4 4 4" = 4265, "8 8" = 7597);
conv is ~9× cheaper per (k, width), so full-span k=L is affordable; the star product
multiplies 6 masked features (vs 4) → sharper nonlinearity, watch the guard; under
OBC only (L−2)³ stars are full (8/64 at L=4) — boundary tokens are weakly pinned.

Runtime: L=4 ≈ 4 s/step on an A100 (~15 min per 200-iter run); on a T4 set
`chunk_size=1024` in BASE and expect ~3–4× slower.


In [ ]:
# ============================== §1 CONFIG ====================================
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
BRANCH   = "feat/dual-basis"

BC            = "OBC"
OUT_ROOT      = "outputs/dual_basis_colab"   # per-run JSON/mpack/curve/log land here
SKIP_EXISTING = True                         # finished {name}.json -> skip (resume-safe)

WANDB         = True                         # live logging to wandb.ai (login in §2)
WANDB_PROJECT = "approx-sym-3D-TC"           # train.py default; override if desired
WANDB_ENTITY  = None                         # None -> train.py default entity

# shared sampling config (campaign standard); dt/diag_shift/arch are PER-CELL
BASE = dict(noninv_channels=4, n_noninv=2,
            n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048, qgt="dense")


In [ ]:
# ============================== §2 SETUP =====================================
import os, sys, json, glob, subprocess
if not os.path.isdir("repo"):
    !git clone --branch $BRANCH $REPO_URL repo
else:
    !cd repo && git fetch origin $BRANCH && git checkout $BRANCH && git pull
%cd repo
%pip -q install netket flax optax wandb

import jax
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), \
    "no GPU — Runtime > Change runtime type > GPU (and re-run this cell)"

if WANDB:
    import wandb
    wandb.login()          # paste your wandb.ai API key when prompted


In [ ]:
# ============================== §3 helpers ===================================
import numpy as np

def run(name, *, dual, L, hx, hz, n_iter, dt, lr_min, diag_shift,
        kernel_size, inv_hidden, seed=0, group=None, subdir="tune", **extra):
    """One independent training run (fresh subprocess -> fresh JAX memory).

    All optimizer/architecture hyperparameters are EXPLICIT arguments — nothing
    hidden except the shared sampling config in BASE (override via **extra).
    Logs live to W&B (group = subdir by default) and streams stdout into the cell
    so you watch per-step energy/spread as it trains. Skips finished runs.
    """
    out_dir = f"{OUT_ROOT}/{subdir}_L{L}"
    jp = f"{out_dir}/{name}.json"
    if SKIP_EXISTING and os.path.exists(jp):
        print(f"[skip] {name} (done)"); return jp
    cfg = {**BASE, **extra}
    cmd = [sys.executable, "-u", "-m", "Three_TC.train",
           "--L", str(L), "--bc", BC, "--arch", "ToricCNN_gridinv",
           "--hx", str(hx), "--hz", str(hz), "--n_iter", str(n_iter),
           "--dt", str(dt), "--lr_min", str(lr_min), "--diag_shift", str(diag_shift),
           "--seed", str(seed), "--inv_hidden", *str(inv_hidden).split(),
           "--out_dir", out_dir, "--name", name, "--resume",
           "--wandb_project", WANDB_PROJECT,
           "--wandb_group", group or f"{subdir}_L{L}"]
    if dual:            cmd += ["--dual_basis"]
    if kernel_size:     cmd += ["--kernel_size", str(kernel_size)]
    if not WANDB:       cmd += ["--no_wandb"]
    if WANDB_ENTITY:    cmd += ["--wandb_entity", WANDB_ENTITY]
    for k, v in cfg.items():
        cmd += [f"--{k}", str(v)]
    os.makedirs(out_dir, exist_ok=True)
    with open(f"{out_dir}/{name}.log", "w") as lg:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:                      # stream into the cell AND the log
            lg.write(line)
            if line.startswith(("[train]", "  step", "  [guard]", "  [t] ")):
                print(line, end="", flush=True)
    if p.wait() != 0:
        print(f"NONZERO EXIT ({p.returncode}) — see {out_dir}/{name}.log")
    return jp

def n_params(**cfg):
    """Cheap parameter count: build the ansatz + init once, no training."""
    from Three_TC.builders import with_defaults, build_geometry, build_model
    c = with_defaults(dict(cfg)); geo = build_geometry(c)
    p = build_model(c, geo).init(jax.random.PRNGKey(0), np.ones((1, geo.N)))
    return sum(int(np.prod(np.shape(l))) for l in jax.tree_util.tree_leaves(p))

def load_all(subdir, L):
    docs = []
    for jp in sorted(glob.glob(f"{OUT_ROOT}/{subdir}_L{L}/*.json")):
        if jp.endswith(".curve.json"): continue
        with open(jp) as f: d = json.load(f)
        log = jp[:-5] + ".log"
        d["n_rollbacks"] = (open(log).read().count("[guard]") if os.path.exists(log) else 0)
        docs.append(d)
    return docs

def row(d):
    o, c = d["observables"], d["config"]
    sp = np.asarray(d["curve"]["energy_spread"], float)
    late = sp[-max(1, len(sp)//5):]
    return dict(name=d["name"], dual=bool(c.get("dual_basis")), E=o["E0"], E_err=o["E_err"],
                Vscore=o["Vscore"], A_v=o["A_v_mean"], B_p=o["B_p_mean"], M_z=o["sz_mean"],
                spread_late=float(np.median(late)),
                spread_spike=float(np.max(sp) / (np.median(sp) + 1e-30)),
                rollbacks=d["n_rollbacks"], diverged=bool(d.get("diverged")),
                n_params=c.get("n_params"),
                s_per_step=d["runtime_s"] / max(1, len(sp)))


## Part 1 — dual-net tuning at L=4, one run per cell

All runs at the same hx-dominated point (hx=0.7, hz=0.2 — near the L=4 transition,
mag consensus ≈0.67). Every hyperparameter is spelled out in each cell; edit freely
and re-run a single cell to redo that one experiment (`SKIP_EXISTING` skips finished
names — bump the name when you change knobs). All runs land in W&B group `tune_L4`.

Reading guide: **kernel** — full span k=L is ~9× cheaper here than for the primal, so
it's the default; k=3 tests whether depth-stacked receptive field suffices. **widths**
— "4 4 4" ≈ 0.8× primal params, "8 8" ≈ 1.4×. **diag_shift/dt** — at an hx point the
dual net is in its diagonal regime so quiet-regime settings should transfer, but the
6-feature star product is sharper than the primal's 4-product; if energy staircases or
`[guard]` lines appear, the stiffer shift is the fix. **noninv_channels** — the star
product is per-channel, so the invariant block's input width == C (the primal gets 3C
from orientation folding); C is the two-sided lever that restores token diversity
(dual G: C=8 + inv"2 2 2" = 4959 params, the closest primal match at 0.93×).


In [ ]:
# --- anchor: PRIMAL production config, same point/budget ----------------------
run("primal_ref", dual=False, L=4, hx=0.7, hz=0.2,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3,
    kernel_size=3, inv_hidden="2 2 2")           # 5319 params


In [ ]:
# --- dual A: full-span kernel, matched-ish capacity (the default guess) -------
run("dual_k4_inv444", dual=True, L=4, hx=0.7, hz=0.2,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3,
    kernel_size=None, inv_hidden="4 4 4")        # k=None -> L=4; 4265 params (0.80x)


In [ ]:
# --- dual B: smaller kernel — is depth-stacked receptive field enough? --------
run("dual_k3_inv444", dual=True, L=4, hx=0.7, hz=0.2,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3,
    kernel_size=3, inv_hidden="4 4 4")           # 2341 params (0.44x)


In [ ]:
# --- dual C: capacity floor (primal-matched widths, NOT params) ---------------
run("dual_k4_inv222", dual=True, L=4, hx=0.7, hz=0.2,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3,
    kernel_size=None, inv_hidden="2 2 2")        # ~1400 params (0.27x)


In [ ]:
# --- dual D: wide + shallow (params overshoot) --------------------------------
run("dual_k4_inv88", dual=True, L=4, hx=0.7, hz=0.2,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3,
    kernel_size=None, inv_hidden="8 8")          # 7597 params (1.43x)


In [ ]:
# --- dual E: stiffer SR (if the 6-product makes gradients spiky) ---------------
run("dual_k4_inv444_ds5e3", dual=True, L=4, hx=0.7, hz=0.2,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=5e-3,
    kernel_size=None, inv_hidden="4 4 4")


In [ ]:
# --- dual F: hotter learning rate ----------------------------------------------
run("dual_k4_inv444_dt02", dual=True, L=4, hx=0.7, hz=0.2,
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3,
    kernel_size=None, inv_hidden="4 4 4")


In [ ]:
# --- dual G: capacity in the TOKENS instead of the invariant block --------------
# The star product is per-channel, so invariant input channels == noninv_channels
# (the primal gets 3x that for free from orientation folding). Raising C restores
# token diversity rather than conv width — and C=8 + inv"2 2 2" is the best param
# match to the primal (4959 vs 5319, ratio 0.93).
run("dual_k4_inv222_c8", dual=True, L=4, hx=0.7, hz=0.2,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3,
    kernel_size=None, inv_hidden="2 2 2", noninv_channels=8)


In [ ]:
# ============================== Part 1 analysis ==============================
import matplotlib.pyplot as plt
import pandas as pd
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})

docs = load_all("tune", 4)
tab = pd.DataFrame([row(d) for d in docs]).sort_values("E")
display(tab.round(6))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for d in docs:
    c, ref = d["curve"], d["name"] == "primal_ref"
    kw = dict(color="k", ls="--", lw=1.6, zorder=5) if ref else dict(lw=1.2)
    ax[0].plot(c["step"], c["energy"], label=d["name"], **kw)
    ax[1].semilogy(c["step"], c["energy_spread"], **kw)
ax[0].set(xlabel="step", ylabel="E", title="dual tuning vs primal (--k): L=4, hx=0.7, hz=0.2")
ax[1].set(xlabel="step", ylabel="energy spread  $\\sqrt{Var[H]}$", title="stability")
ax[0].legend(fontsize=7, loc="upper left")
fig.tight_layout()
# W&B: same curves live under group tune_L4 — the table here adds rollbacks/spike/params.


## Part 2 — primal vs dual A/B (the North Star)

Same points, same seeds, matched parameter budget (check with the cell below; widen the
dual's `inv_hidden` until the ratio is ~0.9–1.1). One cell per field point; each runs
both arms × seeds with fully explicit hyperparameters — carry over the Part-1 winner's
optimizer settings to the dual arm. W&B group `ab_L4`.


In [ ]:
# ============================== param match ==================================
common = dict(L=4, bc=BC, arch="ToricCNN_gridinv", noninv_channels=4, n_noninv=2)
np_primal = n_params(**common, kernel_size=3, inv_hidden=[2, 2, 2])
np_dual   = n_params(**common, dual_basis=True, kernel_size=None, inv_hidden=[4, 4, 4])
print(f"n_params  primal={np_primal}  dual={np_dual}  ratio={np_dual/np_primal:.2f}")


In [ ]:
# --- point 1: hx=0.4, hz=0.2 (hx cut, inside topological) ----------------------
for seed in (0, 1):
    run(f"primal_hx0.4_hz0.2_s{seed}", dual=False, L=4, hx=0.4, hz=0.2,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3,
        kernel_size=3, inv_hidden="2 2 2", seed=seed, subdir="ab")
    run(f"dual_hx0.4_hz0.2_s{seed}", dual=True, L=4, hx=0.4, hz=0.2,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3,
        kernel_size=None, inv_hidden="4 4 4", seed=seed, subdir="ab")


In [ ]:
# --- point 2: hx=0.7, hz=0.2 (hx cut, near the L=4 transition) ------------------
for seed in (0, 1):
    run(f"primal_hx0.7_hz0.2_s{seed}", dual=False, L=4, hx=0.7, hz=0.2,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3,
        kernel_size=3, inv_hidden="2 2 2", seed=seed, subdir="ab")
    run(f"dual_hx0.7_hz0.2_s{seed}", dual=True, L=4, hx=0.7, hz=0.2,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3,
        kernel_size=None, inv_hidden="4 4 4", seed=seed, subdir="ab")


In [ ]:
# --- point 3: hx=0.2, hz=0.36 (hz CONTROL — the primal's diagonal regime) -------
for seed in (0, 1):
    run(f"primal_hx0.2_hz0.36_s{seed}", dual=False, L=4, hx=0.2, hz=0.36,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3,
        kernel_size=3, inv_hidden="2 2 2", seed=seed, subdir="ab")
    run(f"dual_hx0.2_hz0.36_s{seed}", dual=True, L=4, hx=0.2, hz=0.36,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3,
        kernel_size=None, inv_hidden="4 4 4", seed=seed, subdir="ab")


In [ ]:
# ============================== Part 2 analysis ==============================
docs = load_all("ab", 4)
def _pt(d):
    c = d["config"]; return f"hx{c['hx']}_hz{c['hz']}"
tab = pd.DataFrame([dict(point=_pt(d), arm=("dual" if d["config"].get("dual_basis")
                                            else "primal"), seed=d["config"]["seed"],
                         **row(d)) for d in docs])
display(tab.drop(columns=["name", "dual"]).round(6))

pts = sorted(tab.point.unique())
fig, axes = plt.subplots(2, len(pts), figsize=(4.2 * len(pts), 7), squeeze=False)
ARM = {"primal": dict(color="0.35"), "dual": dict(color=plt.cm.plasma(0.65))}
for j, p in enumerate(pts):
    for d in docs:
        if _pt(d) != p: continue
        a = "dual" if d["config"].get("dual_basis") else "primal"
        c = d["curve"]
        axes[0][j].plot(c["step"], c["energy"], "-", lw=1.2, alpha=0.9,
                        label=f"{a} s{d['config']['seed']}", **ARM[a])
        axes[1][j].semilogy(c["step"], c["energy_spread"], "-", lw=1.2, **ARM[a])
    axes[0][j].set(title=p, xlabel="step", ylabel="E" if j == 0 else None)
    axes[1][j].set(xlabel="step", ylabel="spread" if j == 0 else None)
    axes[0][j].legend(fontsize=7, loc="upper right")
fig.tight_layout()

v = (tab.groupby(["point", "arm"])
        .agg(E=("E", "mean"), Vscore=("Vscore", "mean"),
             rollbacks=("rollbacks", "sum"), diverged=("diverged", "any"))
        .reset_index())
print("\n=== verdict (lower E wins; check Vscore + stability alongside) ===")
display(v.round(6))
for p in pts:   # physical-consistency gate: arms measure the SAME observables
    sub = tab[tab.point == p].groupby("arm")[["A_v", "B_p", "M_z"]].mean()
    if len(sub) == 2 and (sub.diff().abs().iloc[-1] > 0.05).any():
        print(f"WARNING {p}: arms disagree on a physical observable\n{sub}")
